In [36]:
from lcdb.db import LCDB
from lcdb.analysis import LearningCurveExtractor
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from lcdb.analysis import merge_curves
import pandas as pd
from lcdb.analysis import LearningCurveGroup



In [37]:


workflows = []
workflows.append('lcdb.workflow.sklearn.KNNWorkflow')
workflows.append('lcdb.workflow.sklearn.LibLinearWorkflow')
workflows.append('lcdb.workflow.sklearn.LibSVMWorkflow')

workflow = np.random.choice(workflows)

if workflow == 'lcdb.workflow.sklearn.KNNWorkflow':
    datasets_with_errors = [41138, 41159, 41161, 42746, 43072]
if workflow == 'lcdb.workflow.sklearn.LibLinearWorkflow':
    datasets_with_errors = [41147, 41165, 42746, 43072]
if workflow == 'lcdb.workflow.sklearn.LibSVMWorkflow':
    datasets_with_errors = [1596, 23517, 40996, 41166, 42746, 43072]

dataset = np.random.choice(datasets_with_errors)

In [38]:
import tqdm
import traceback


try:
      lcdb = LCDB()

      df_full = lcdb.query(
      workflows=[workflow],
      campaigns=['pre-config-100'],
      openmlids=[dataset],
      return_generator=False,
      processors={
            "learning_curve": LearningCurveExtractor(
                  metrics=["error_rate"],
                  folds=["val"]
            )
      },
      show_progress=False
      )

      df = df_full[df_full["learning_curve"].notnull()] # remove empty curves
      config_cols = [c for c in df.columns if c.startswith("p:")]
      df_grouped = df.groupby(config_cols).agg({"learning_curve": merge_curves})
      lcg = LearningCurveGroup(df_grouped["learning_curve"])

      lc_list = []
      for lc in lcg:
            lc_list.append(lc)

      lcs = np.concatenate(lc_list)

      lcs2 = np.squeeze(lcs)
      lcs2.shape
      print('Completed a workflow')

except Exception as e:
      print(f"For workflow {workflow} on dataset {dataset}")
      print(f"An error occurred: {type(e).__name__}: {e}")
      traceback.print_exc()


For workflow lcdb.workflow.sklearn.LibSVMWorkflow on dataset 1596
An error occurred: NotImplementedError: 


Traceback (most recent call last):
  File "/tmp/ipykernel_16712/2061205887.py", line 25, in <module>
    lcg = LearningCurveGroup(df_grouped["learning_curve"])
  File "/home/tom/projects/lcdb/publications/2023-neurips/lcdb/analysis/_learning_curves.py", line 151, in __init__
    self.add_curve(curve)
  File "/home/tom/projects/lcdb/publications/2023-neurips/lcdb/analysis/_learning_curves.py", line 205, in add_curve
    raise NotImplementedError
NotImplementedError
